# AutoML Test — benchmark exploratorio temporal

Este notebook prueba varios modelos y preprocesados sobre la misma tabla integrada. Es un laboratorio para comparar métricas; no sustituye `Models.ipynb`, no usa AutoML para producción y no guarda ni promociona modelos en `models/`.


## Cómo leer este experimento

- Las configuraciones se comparan primero en un periodo de validación futuro: 2023.
- Solo las tres mejores se reentrenan con datos hasta 2023 y se miden en 2024.
- Nunca hay división aleatoria: todo respeta el orden temporal.
- Un MAE menor no basta para adoptar un modelo: habría que revisar estabilidad, explicabilidad, fuga de información y sentido de negocio.


In [1]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler
from sklearn.svm import SVR

RANDOM_SEED = 42
TARGET = 'avg_duration_months'
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / 'data' / 'processed').exists() else cwd.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'train_features.csv'
print(f'Project root: {PROJECT_ROOT}')


Project root: /home/kiril/Proyectos/starwars_autocalls


In [2]:
data = pd.read_csv(DATA_PATH, parse_dates=['requested_date', 'start_date', 'end_date'])

FORBIDDEN_COLUMNS = {
    'rfq_id', TARGET, 'executed', 'underlyings', 'observation_frequency',
    'requested_date', 'start_date', 'end_date',
}
# Keep this benchmark aligned with the approved primary-model contract.
EXPERIMENTAL_ONLY_COLUMNS = {'realized_vol_trend_21d_mean'}
feature_columns = [
    column for column in data.columns
    if column not in FORBIDDEN_COLUMNS and column not in EXPERIMENTAL_ONLY_COLUMNS
]

assert data['executed'].eq(True).all()
assert data[TARGET].notna().all()
assert not set(feature_columns) & FORBIDDEN_COLUMNS
assert not data[feature_columns].isna().any().any()
print(f'{len(data):,} RFQs ejecutadas y {len(feature_columns)} variables sin fuga.')


13,796 RFQs ejecutadas y 96 variables sin fuga.


## Protocolo temporal

Usamos 2023 para elegir configuraciones y 2024 para una comprobación final de las tres mejores. Este último periodo ya fue usado en `Models.ipynb`, así que aquí se trata como una comprobación exploratoria, no como un nuevo resultado de producción independiente.


In [3]:
validation_start = pd.Timestamp('2023-01-01')
test_start = pd.Timestamp('2024-01-01')

train = data.loc[data['requested_date'] < validation_start].copy()
validation = data.loc[(data['requested_date'] >= validation_start) & (data['requested_date'] < test_start)].copy()
test = data.loc[data['requested_date'] >= test_start].copy()

assert train.requested_date.max() < validation.requested_date.min() < test.requested_date.min()
split_summary = pd.DataFrame({
    'split': ['train', 'validation', 'test exploratorio'],
    'n_rfqs': [len(train), len(validation), len(test)],
    'from': [train.requested_date.min().date(), validation.requested_date.min().date(), test.requested_date.min().date()],
    'to': [train.requested_date.max().date(), validation.requested_date.max().date(), test.requested_date.max().date()],
})
display(split_summary)


,split,n_rfqs,from,to
0,train,11385,2016-01-04,2022-12-30
1,validation,1665,2023-01-02,2023-12-29
2,test exploratorio,746,2024-01-01,2024-06-28


## Variables y preprocesados probados

Se comparan dos contratos de variables:

- **Todas:** las 96 variables disponibles al cotizar.
- **Compactas:** elimina tres columnas redundantes de control de calidad (`n_underlyings`, `n_market_matches`, `market_match_rate`).

Los modelos lineales y de distancia necesitan escalar variables. Los árboles no: sus particiones no dependen de que una variable esté en euros, meses o porcentaje.


In [4]:
redundant_columns = ['n_underlyings', 'n_market_matches', 'market_match_rate']
all_features = feature_columns.copy()
compact_features = [column for column in feature_columns if column not in redundant_columns]

dummy_prefixes = ('has_underlying_', 'product_type_', 'basket_type_', 'counterparty_', 'trader_id_')

def feature_groups(columns: list[str]) -> tuple[list[str], list[str]]:
    dummies = [column for column in columns if column.startswith(dummy_prefixes)]
    continuous = [column for column in columns if column not in dummies]
    return continuous, dummies

def mixed_scaler(columns: list[str], scaler) -> ColumnTransformer:
    continuous, dummies = feature_groups(columns)
    return ColumnTransformer([
        ('continuous', scaler, continuous),
        ('dummies', 'passthrough', dummies),
    ], remainder='drop')

print(f'Todas: {len(all_features)} variables | Compactas: {len(compact_features)} variables')


Todas: 96 variables | Compactas: 93 variables


## Batería de configuraciones

No es una búsqueda infinita. Es una parrilla pequeña, explícita y reproducible: referencias simples, modelos lineales, modelos por distancia, ensembles de árboles, boosting y CatBoost.


In [5]:
def ridge_standard(columns):
    return Pipeline([('preprocess', mixed_scaler(columns, StandardScaler())), ('model', Ridge(alpha=10.0))])

def ridge_robust(columns):
    return Pipeline([('preprocess', mixed_scaler(columns, RobustScaler())), ('model', Ridge(alpha=10.0))])

def elastic_net(columns):
    return Pipeline([('preprocess', mixed_scaler(columns, StandardScaler())), ('model', ElasticNet(alpha=0.03, l1_ratio=0.15, max_iter=10_000, random_state=RANDOM_SEED))])

def knn_standard(columns):
    return Pipeline([('scale', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=30, weights='distance', p=2, n_jobs=-1))])

def svr_standard(columns):
    return Pipeline([('scale', MinMaxScaler()), ('model', SVR(C=10.0, epsilon=0.2, gamma='scale'))])

def catboost_model():
    return CatBoostRegressor(
        loss_function='MAE', eval_metric='MAE', iterations=800, learning_rate=0.05,
        depth=8, l2_leaf_reg=5.0, random_seed=RANDOM_SEED,
        verbose=False, allow_writing_files=False,
    )

configs = [
    {'name': 'Median reference', 'features': all_features, 'preprocess': 'none', 'factory': lambda columns: DummyRegressor(strategy='median')},
    {'name': 'Ridge standard / all', 'features': all_features, 'preprocess': 'standard continuous', 'factory': ridge_standard},
    {'name': 'Ridge robust / compact', 'features': compact_features, 'preprocess': 'robust continuous', 'factory': ridge_robust},
    {'name': 'ElasticNet / compact', 'features': compact_features, 'preprocess': 'standard continuous', 'factory': elastic_net},
    {'name': 'KNN / compact', 'features': compact_features, 'preprocess': 'standard all', 'factory': knn_standard},
    {'name': 'SVR RBF / compact', 'features': compact_features, 'preprocess': 'min-max all', 'factory': svr_standard},
    {'name': 'Random forest / all', 'features': all_features, 'preprocess': 'none', 'factory': lambda columns: RandomForestRegressor(n_estimators=350, min_samples_leaf=3, max_features=0.7, random_state=RANDOM_SEED, n_jobs=-1)},
    {'name': 'Random forest / compact', 'features': compact_features, 'preprocess': 'none', 'factory': lambda columns: RandomForestRegressor(n_estimators=350, min_samples_leaf=3, max_features=0.7, random_state=RANDOM_SEED, n_jobs=-1)},
    {'name': 'Extra trees / all', 'features': all_features, 'preprocess': 'none', 'factory': lambda columns: ExtraTreesRegressor(n_estimators=350, min_samples_leaf=2, max_features=0.8, random_state=RANDOM_SEED, n_jobs=-1)},
    {'name': 'Extra trees / compact', 'features': compact_features, 'preprocess': 'none', 'factory': lambda columns: ExtraTreesRegressor(n_estimators=350, min_samples_leaf=2, max_features=0.8, random_state=RANDOM_SEED, n_jobs=-1)},
    {'name': 'Gradient boosting / compact', 'features': compact_features, 'preprocess': 'none', 'factory': lambda columns: GradientBoostingRegressor(n_estimators=300, learning_rate=0.04, max_depth=3, min_samples_leaf=8, loss='huber', random_state=RANDOM_SEED)},
    {'name': 'Histogram gradient boosting / all', 'features': all_features, 'preprocess': 'none', 'factory': lambda columns: HistGradientBoostingRegressor(max_iter=350, learning_rate=0.05, max_leaf_nodes=31, l2_regularization=2.0, random_state=RANDOM_SEED)},
    {'name': 'CatBoost / all', 'features': all_features, 'preprocess': 'none', 'factory': lambda columns: catboost_model()},
]

pd.DataFrame([{key: value for key, value in config.items() if key != 'factory'} for config in configs])


,name,features,preprocess
0,Median reference,"[autocall_barrier_pct, protection_barrier_pct,...",none
1,Ridge standard / all,"[autocall_barrier_pct, protection_barrier_pct,...",standard continuous
2,Ridge robust / compact,"[autocall_barrier_pct, protection_barrier_pct,...",robust continuous
3,ElasticNet / compact,"[autocall_barrier_pct, protection_barrier_pct,...",standard continuous
4,KNN / compact,"[autocall_barrier_pct, protection_barrier_pct,...",standard all
5,SVR RBF / compact,"[autocall_barrier_pct, protection_barrier_pct,...",min-max all
6,Random forest / all,"[autocall_barrier_pct, protection_barrier_pct,...",none
7,Random forest / compact,"[autocall_barrier_pct, protection_barrier_pct,...",none
8,Extra trees / all,"[autocall_barrier_pct, protection_barrier_pct,...",none
9,Extra trees / compact,"[autocall_barrier_pct, protection_barrier_pct,...",none


In [6]:
def evaluate_configs(configs, train_frame, evaluation_frame, label: str) -> pd.DataFrame:
    rows = []
    for config in configs:
        started = time.perf_counter()
        try:
            model = config['factory'](config['features'])
            model.fit(train_frame[config['features']], train_frame[TARGET])
            prediction = model.predict(evaluation_frame[config['features']])
            rows.append({
                'name': config['name'], 'preprocess': config['preprocess'],
                'n_features': len(config['features']), 'status': 'ok',
                f'{label}_MAE_months': mean_absolute_error(evaluation_frame[TARGET], prediction),
                f'{label}_RMSE_months': root_mean_squared_error(evaluation_frame[TARGET], prediction),
                'fit_seconds': time.perf_counter() - started,
            })
        except Exception as error:
            rows.append({
                'name': config['name'], 'preprocess': config['preprocess'],
                'n_features': len(config['features']), 'status': f'error: {type(error).__name__}',
                'fit_seconds': time.perf_counter() - started,
            })
    return pd.DataFrame(rows)

validation_results = evaluate_configs(configs, train, validation, 'validation')
validation_results = validation_results.sort_values('validation_MAE_months', na_position='last').reset_index(drop=True)
print(validation_results.to_string(index=False, float_format=lambda value: f'{value:.3f}'))
display(validation_results.style.format({
    'validation_MAE_months': '{:.3f}', 'validation_RMSE_months': '{:.3f}', 'fit_seconds': '{:.2f}',
}))


                             name          preprocess  n_features status  validation_MAE_months  validation_RMSE_months  fit_seconds
              Random forest / all                none          96     ok                 11.219                  15.043        1.677
          Random forest / compact                none          93     ok                 11.239                  15.095        1.693
Histogram gradient boosting / all                none          96     ok                 11.257                  14.940        0.332
                   CatBoost / all                none          96     ok                 11.337                  15.234        2.448
      Gradient boosting / compact                none          93     ok                 11.359                  15.107        9.122
            Extra trees / compact                none          93     ok                 11.439                  15.385        1.588
                Extra trees / all                none          96    

,name,preprocess,n_features,status,validation_MAE_months,validation_RMSE_months,fit_seconds
0,Random forest / all,none,96,ok,11.219,15.043,1.68
1,Random forest / compact,none,93,ok,11.239,15.095,1.69
2,Histogram gradient boosting / all,none,96,ok,11.257,14.940,0.33
3,CatBoost / all,none,96,ok,11.337,15.234,2.45
4,Gradient boosting / compact,none,93,ok,11.359,15.107,9.12
5,Extra trees / compact,none,93,ok,11.439,15.385,1.59
6,Extra trees / all,none,96,ok,11.473,15.426,1.62
7,SVR RBF / compact,min-max all,93,ok,12.042,15.732,5.13
8,Ridge robust / compact,robust continuous,93,ok,12.259,15.855,0.03
9,Ridge standard / all,standard continuous,96,ok,12.259,15.853,0.03


## Comprobación final de las tres mejores configuraciones

La selección se hace exclusivamente con 2023. Las tres primeras se vuelven a entrenar usando todo el histórico hasta el 31 de diciembre de 2023 y se miden en 2024.


In [7]:
top_names = validation_results.loc[validation_results['status'].eq('ok'), 'name'].head(3).tolist()
top_configs = [config for config in configs if config['name'] in top_names]
train_plus_validation = data.loc[data['requested_date'] < test_start].copy()

test_results = evaluate_configs(top_configs, train_plus_validation, test, 'test')
test_results = test_results.sort_values('test_MAE_months', na_position='last').reset_index(drop=True)
print(test_results.to_string(index=False, float_format=lambda value: f'{value:.3f}'))
display(test_results.style.format({
    'test_MAE_months': '{:.3f}', 'test_RMSE_months': '{:.3f}', 'fit_seconds': '{:.2f}',
}))

print('Este ranking es exploratorio. Ningún resultado de este notebook modifica el modelo principal guardado en models/.')


                             name preprocess  n_features status  test_MAE_months  test_RMSE_months  fit_seconds
          Random forest / compact       none          93     ok           11.257            15.150        2.003
              Random forest / all       none          96     ok           11.318            15.208        1.977
Histogram gradient boosting / all       none          96     ok           11.470            15.158        0.402


,name,preprocess,n_features,status,test_MAE_months,test_RMSE_months,fit_seconds
0,Random forest / compact,none,93,ok,11.257,15.150,2.00
1,Random forest / all,none,96,ok,11.318,15.208,1.98
2,Histogram gradient boosting / all,none,96,ok,11.470,15.158,0.40


Este ranking es exploratorio. Ningún resultado de este notebook modifica el modelo principal guardado en models/.


## Qué hacer con los resultados

Este notebook sirve para saciar curiosidad y encontrar candidatos. Si uno de los modelos fuese claramente mejor, el siguiente paso no sería promoverlo: habría que repetir la comparación con backtesting temporal, revisar sus variables y asegurar que sigue teniendo sentido de negocio.
